In [8]:
pip install openpyxl


Looking in indexes: https://pypi.org/simple, https://pypi.ngc.nvidia.com

   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   ---------------------------------------- 2/2 [openpyxl]

Note: you may need to restart the kernel to use updated packages.


## Label Statistics Summary

Reads `labels.xlsx` from the project root, prints the first few rows, and computes label counts and label prevalence (mean) for all label columns (excluding `id`).


In [1]:
import pandas as pd

df = pd.read_excel("labels.xlsx")
print(df.head())

label_counts = df.drop(columns=["id"]).sum()
print(label_counts)

label_means = df.drop(columns=["id"]).mean()
print(label_means)


   id  消化系统疾病  内分泌代谢疾病  泌尿生殖系统疾病  血液及造血器官疾病和涉及免疫机制的某些疾患
0   3       0        1         0                      0
1   4       0        1         0                      0
2   5       1        0         0                      0
3   6       0        0         0                      0
4   9       0        0         0                      0
消化系统疾病                    31
内分泌代谢疾病                  111
泌尿生殖系统疾病                  33
血液及造血器官疾病和涉及免疫机制的某些疾患     20
dtype: int64
消化系统疾病                   0.146919
内分泌代谢疾病                  0.526066
泌尿生殖系统疾病                 0.156398
血液及造血器官疾病和涉及免疫机制的某些疾患    0.094787
dtype: float64


## Encoder Features Export

- **Input (project root):** `encoder_instance_features.xlsx`
- **Output (project root):** `encoder_features/`

Each `.pt` file contains a feature tensor of shape `[N, 768]`, where:
- **N** = number of colonoscopy images for that patient  
- **768** = DINOv2 feature dimension


In [ ]:
encoder_instance_features.xlsx
↓
(group by patient_id)
↓
encoder_features/
├── patient_id1.pt
├── patient_id2.pt
└── ...

In [1]:
import pandas as pd
import torch
import os

df = pd.read_excel("encoder_instance_features.xlsx")

feature_cols = [c for c in df.columns if c.startswith("feat_")]


for pid, group in df.groupby("patient_id"):
    feats = group[feature_cols].values  # shape [N, D]
    feats = torch.tensor(feats, dtype=torch.float32)

    torch.save(
        {"patient_id": pid, "feats": feats},
        f"encoder_features/{pid}.pt"
    )

print("DONE — saved per-patient features in features_pt/")


DONE — saved per-patient features in features_pt/




## Training Summary

Trains an attention-based patient-level model using per-patient encoder features (`encoder_features/`) to predict **内分泌代谢疾病**, with random instance sampling, combined BCE+AUC loss, 5-fold cross-validation, and saves training logs to the **outputs** folder.



In [1]:
%run train_log.py \
  --feat_dir ./encoder_features \
  --labels_csv ./labels.xlsx \
  --label_cols 内分泌代谢疾病 \
  --max_feats 16 \
  --epochs 100 \
  --batch_size 4 \
  --lr 5e-4 \
  --weight_decay 1e-4 \
  --folds 5 \
  --instance_strategy random \
  --architecture attention \
  --use_combined_loss \
  --auc_weight 0.5 

===== TRAINING START =====
feat_dir: ./encoder_features
labels_csv: ./labels.xlsx
label_cols: ['内分泌代谢疾病']
epochs: 100
batch_size: 4
lr: 0.0005
weight_decay: 0.0001
max_feats: 16
folds: 5
num_workers: 0
instance_strategy: random
architecture: attention
use_combined_loss: True
auc_weight: 0.5
seed: 42

Single-label task: StratifiedKFold
Using device: cuda

===== Fold 1/5 =====
Epoch 001: loss=0.3982  AUROC=0.4826  AUPRC=0.5210  Sens@95Spec=0.0000  Brier=0.2658  ✓ NEW BEST
Epoch 002: loss=0.3832  AUROC=0.5543  AUPRC=0.6253  Sens@95Spec=0.1304  Brier=0.2546  ✓ NEW BEST
Epoch 003: loss=0.3781  AUROC=0.4978  AUPRC=0.5675  Sens@95Spec=0.0435  Brier=0.2544
Epoch 004: loss=0.3657  AUROC=0.4783  AUPRC=0.4994  Sens@95Spec=0.0000  Brier=0.2627
Epoch 005: loss=0.3587  AUROC=0.4370  AUPRC=0.5604  Sens@95Spec=0.1304  Brier=0.2747
Epoch 006: loss=0.3586  AUROC=0.5196  AUPRC=0.5639  Sens@95Spec=0.0435  Brier=0.2621
Epoch 007: loss=0.3520  AUROC=0.4261  AUPRC=0.5027  Sens@95Spec=0.0000  Brier=0.2814
Epo